# DDL: `dbspend360_pool_spends`

Creates the per-pool spend table for **Databricks instance pools**.
Populated by `dbspend360_pool_spends_app`.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

**Schema notes**
- `idle_cloud_cost = pool_total_cost - active_cloud_cost`, floored at 0 to absorb
  CE/Azure-CM tag-collection lag (see
  `docs/plans/shared_clusters_and_pools/05-slice-3-instance-pools.md` §`greatest(...)`).
- `databricks_cost` is non-zero only on premium-edition pool surcharges; cluster
  runtime DBU bills to the cluster, not the pool.
- Compute / storage / network split is intentionally dropped: pool VMs are pure
  compute so the split adds no signal.
- Merge key: `(instance_pool_id, workspace_id, usage_date)`.

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_pool_spends (
  instance_pool_id   STRING,
  workspace_id       STRING,
  usage_date         DATE,
  idle_cloud_cost    DOUBLE,
  active_cloud_cost  DOUBLE,
  pool_total_cost    DOUBLE,
  databricks_cost    DOUBLE,
  currency           STRING,
  total_cost         DOUBLE,
  created_at         TIMESTAMP,
  updated_at         TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_pool_spends")